In [3]:
import heapq
from collections import defaultdict
from typing import List

Running median : return a list of the median so far 

In [4]:
def running_median(self, nums : List[int]) -> List[float] : 
    low = [] # max-heap :  all elements are inferior to the top (we will stock -value)
    high = [] # min-heap :  all elements are superior to the top 
    res = []
    for val in nums : 
        if len(low) == len(high) :
            heapq.heappush(low,-val)
            heapq.heappush(high, -heapq.heappop(low))
        else : 
            heapq.heappush(high,val)
            heapq.heappush(low,-heapq.heappop(high))
        res.append(high[0] if len(high) > len(low) else (high[0]-low[0])/2)
    return res            

# Sliding median of window k
Return a list of median where : 
    result[i] = median(nums[i:i+k])

"Lazy deletion ":
    Instead of deleting from the heap the values that will be out of the window, we are going to mark them and when there will be in top of the heap we will remove them. 
    Since we are not deleting from the heap, we need to keep two variables containing the true size of each heap.

In [6]:
def window_median(self, nums: List[int], k: int) -> List[float]:
    low, high = [], []          
    delayed = defaultdict(int)
    low_size = high_size = 0    

    def prune(heap, sign):
        while heap and delayed[sign * heap[0]] > 0:
            delayed[sign * heap[0]] -= 1
            heapq.heappop(heap)

    def high_to_low():
        nonlocal low_size, high_size
        prune(high, 1)
        heapq.heappush(low, -heapq.heappop(high))
        high_size -= 1
        low_size += 1

    def low_to_high():
        nonlocal low_size, high_size
        prune(low, -1)
        heapq.heappush(high, -heapq.heappop(low))
        low_size -= 1
        high_size += 1

    def median():
        prune(low, -1)
        prune(high, 1)
        return high[0] if k % 2 else (high[0] - low[0]) / 2

    # Push everything in high then send the k//2 smallest into low 
    for x in nums[:k]:
        heapq.heappush(high, x)
    high_size = k
    for _ in range(k // 2):
        high_to_low()

    res = [median()]
    for i in range(k, len(nums)):
        inn, out = nums[i], nums[i - k]

        # Insertion
        if inn >= high[0]:
            heapq.heappush(high, inn)
            high_size += 1
        else:
            heapq.heappush(low, -inn)
            low_size += 1

        # Lazy deletion : increment the count of out & update the size of low or high in consequence
        delayed[out] += 1
        if out >= high[0]:
            high_size -= 1
        else:
            low_size -= 1

        # Requilibrate both of the heaps
        if high_size > low_size + 1:
            high_to_low()
        elif high_size < low_size:
            low_to_high()

        res.append(median())
    return res